# Pattern Recognition, Mock Interviews and a 12-Week Plan

Notebook for **Chapter 12** of the *dsa* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/dsa/ch12.html

Tip: change a number, predict the output, then run the cell again.

## 12.1 The Two-Minute Decision Flow

In [ ]:
SIGNALS = [
    ("sorted", "two pointers or binary search"),
    ("contiguous", "sliding window or prefix sums"),
    ("substring", "sliding window"),
    ("k-th", "heap or quickselect"),
    ("top k", "heap"),
    ("all subsets", "backtracking"),
    ("number of ways", "dynamic programming"),
    ("fewest steps", "BFS"),
    ("prerequisite", "topological sort"),
    ("intervals", "sort then sweep"),
    ("next greater", "monotonic stack"),
    ("duplicate", "hash set"),
]

def hint(statement):
    text = statement.lower()
    return [pattern for word, pattern in SIGNALS if word in text]

assert hint("Find the longest contiguous subarray with sum k") == ["sliding window or prefix sums"]
assert hint("Count the number of ways to climb stairs") == ["dynamic programming"]
assert hint("Return the fewest steps; courses have a prerequisite list") == ["BFS", "topological sort"]
for q in ["Longest substring with no duplicate letters",
          "Top K frequent words",
          "Merge overlapping intervals"]:
    print(q, "->", hint(q))

Longest substring with no duplicate letters -> ['sliding window', 'hash set']
Top K frequent words -> ['heap']
Merge overlapping intervals -> ['sort then sweep']


## 12.2 Array and String Patterns

Five patterns cover most array and string questions. They were taught in Chapters 2, 3 and 4. Here is the one-card summary of each.

In [ ]:
def first_duplicate(a):                 # hash set: remember what you have seen
    seen = set()
    for x in a:
        if x in seen:
            return x
        seen.add(x)
    return None

def two_sum_sorted(a, target):          # two pointers: sorted array, move the side that helps
    l, r = 0, len(a) - 1
    while l < r:
        s = a[l] + a[r]
        if s == target:
            return (l, r)
        if s < target:
            l += 1                      # sum too small: need a bigger left value
        else:
            r -= 1                      # sum too big: need a smaller right value
    return None

def longest_unique(s):                  # sliding window with last-seen index
    last = {}
    left = best = 0
    for right, ch in enumerate(s):
        if ch in last and last[ch] >= left:
            left = last[ch] + 1         # jump the left edge past the repeat
        last[ch] = right
        best = max(best, right - left + 1)
    return best

def range_sum_maker(a):                 # prefix sums: pre[k] = sum of the first k items
    pre = [0]
    for x in a:
        pre.append(pre[-1] + x)
    return lambda i, j: pre[j + 1] - pre[i]     # sum of a[i..j] inclusive

def first_true(lo, hi, ok):             # binary search on the answer: smallest x with ok(x) True
    while lo < hi:
        mid = (lo + hi) // 2
        if ok(mid):
            hi = mid                    # mid works, but maybe something smaller does too
        else:
            lo = mid + 1                # mid fails, so the answer is above it
    return lo

assert first_duplicate([4, 2, 7, 2, 9]) == 2
assert two_sum_sorted([1, 3, 4, 6, 8, 11], 10) == (2, 3)
assert longest_unique("abcabcbb") == 3
assert range_sum_maker([5, 1, 4, 2, 8])(1, 3) == 7
assert first_true(0, 100, lambda x: x * x >= 50) == 8
print(first_duplicate([4, 2, 7, 2, 9]), two_sum_sorted([1, 3, 4, 6, 8, 11], 10),
      longest_unique("abcabcbb"), range_sum_maker([5, 1, 4, 2, 8])(1, 3),
      first_true(0, 100, lambda x: x * x >= 50))

2 (2, 3) 3 7 8


In [ ]:
def sorted_squares(a):
    out = [0] * len(a)
    l, r = 0, len(a) - 1
    for k in range(len(a) - 1, -1, -1):      # fill from the back
        if abs(a[l]) > abs(a[r]):
            out[k] = a[l] * a[l]
            l += 1
        else:
            out[k] = a[r] * a[r]
            r -= 1
    return out

assert sorted_squares([-4, -1, 0, 3, 10]) == [0, 1, 9, 16, 100]
assert sorted_squares([-7, -3, 2, 3, 11]) == [4, 9, 9, 49, 121]
assert sorted_squares([-5, -2]) == [4, 25]
assert sorted_squares([]) == []
print(sorted_squares([-4, -1, 0, 3, 10]))

[0, 1, 9, 16, 100]


In [ ]:
from collections import defaultdict

def count_subarrays(a, k):
    seen = defaultdict(int)
    seen[0] = 1                    # the empty prefix, so subarrays that start at index 0 count
    total = count = 0
    for x in a:
        total += x
        count += seen[total - k]   # earlier prefixes that leave a gap of exactly k
        seen[total] += 1
    return count

def brute(a, k):
    return sum(1 for i in range(len(a)) for j in range(i, len(a)) if sum(a[i:j + 1]) == k)

data = [3, 4, 7, 2, -3, 1, 4, 2]
assert count_subarrays(data, 7) == brute(data, 7) == 4
assert count_subarrays([1, 1, 1], 2) == 2
assert count_subarrays([0, 0, 0], 0) == 6
print(count_subarrays(data, 7), count_subarrays([1, 1, 1], 2), count_subarrays([0, 0, 0], 0))

4 2 6


In [ ]:
from collections import Counter

def min_window(s, t):
    if not t or len(t) > len(s):
        return ""
    need = Counter(t)                  # how many of each letter we still need (may go negative)
    missing = len(t)                   # total letters still needed
    best = (0, 0)                      # (start, end) of the best window so far, end exclusive
    best_len = len(s) + 1
    left = 0
    for right, ch in enumerate(s):
        if need[ch] > 0:
            missing -= 1
        need[ch] -= 1
        while missing == 0:            # the window is valid: try to shrink it
            if right - left + 1 < best_len:
                best_len = right - left + 1
                best = (left, right + 1)
            need[s[left]] += 1
            if need[s[left]] > 0:      # we just dropped a letter we truly need
                missing += 1
            left += 1
    return s[best[0]:best[1]] if best_len <= len(s) else ""

assert min_window("ADOBECODEBANC", "ABC") == "BANC"
assert min_window("a", "a") == "a"
assert min_window("a", "aa") == ""
assert min_window("aab", "ab") == "ab"
print(min_window("ADOBECODEBANC", "ABC"), min_window("aab", "ab"), repr(min_window("a", "aa")))

BANC ab ''


## 12.3 Stack, Linked List and Tree Patterns

In [ ]:
from collections import deque

class ListNode:
    def __init__(self, val, nxt=None):
        self.val, self.next = val, nxt

class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def has_cycle(head):                     # fast and slow pointers
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
        if slow is fast:
            return True
    return False

def depth(node):                         # tree DFS: answer from the children
    if node is None:
        return 0
    return 1 + max(depth(node.left), depth(node.right))

def levels(root):                        # tree BFS: one queue, one level per round
    out, q = [], deque([root] if root else [])
    while q:
        level = []
        for _ in range(len(q)):          # len(q) is frozen at the start of the round
            node = q.popleft()
            level.append(node.val)
            if node.left: q.append(node.left)
            if node.right: q.append(node.right)
        out.append(level)
    return out

c = ListNode(3); b = ListNode(2, c); a = ListNode(1, b)
assert has_cycle(a) is False
c.next = a                               # close the loop
assert has_cycle(a) is True
tree = TreeNode(1, TreeNode(2, TreeNode(4), TreeNode(5)), TreeNode(3, TreeNode(6), TreeNode(7)))
assert depth(tree) == 3 and levels(tree) == [[1], [2, 3], [4, 5, 6, 7]]
print(has_cycle(a), depth(tree), levels(tree))

True 3 [[1], [2, 3], [4, 5, 6, 7]]


In [ ]:
def valid_brackets(s):
    partner = {")": "(", "]": "[", "}": "{"}
    stack = []
    for ch in s:
        if ch in partner:
            if not stack or stack.pop() != partner[ch]:
                return False
        else:
            stack.append(ch)
    return not stack

assert valid_brackets("{[()]}") and valid_brackets("")
assert not valid_brackets("(]") and not valid_brackets(")") and not valid_brackets("((")
print(valid_brackets("{[()]}"), valid_brackets("(]"), valid_brackets(")"))

True False False


In [ ]:
def daily_temperatures(t):
    ans = [0] * len(t)
    stack = []                           # indexes of days still waiting
    for i, temp in enumerate(t):
        while stack and t[stack[-1]] < temp:
            j = stack.pop()
            ans[j] = i - j
        stack.append(i)
    return ans

assert daily_temperatures([73, 74, 75, 71, 69, 72, 76, 73]) == [1, 1, 4, 2, 1, 1, 0, 0]
assert daily_temperatures([5, 4, 3]) == [0, 0, 0]
assert daily_temperatures([3, 3, 3]) == [0, 0, 0]
print(daily_temperatures([73, 74, 75, 71, 69, 72, 76, 73]))

[1, 1, 4, 2, 1, 1, 0, 0]


In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def max_path_sum(root):
    best = float("-inf")
    def gain(node):
        nonlocal best
        if node is None:
            return 0
        left = max(gain(node.left), 0)       # drop a branch that hurts
        right = max(gain(node.right), 0)
        best = max(best, node.val + left + right)    # path that bends here
        return node.val + max(left, right)           # parent may extend one side only
    gain(root)
    return best

tree = TreeNode(-5, TreeNode(8), TreeNode(12, TreeNode(10), TreeNode(-4)))
assert max_path_sum(tree) == 25
assert max_path_sum(TreeNode(-3)) == -3
assert max_path_sum(TreeNode(-3, TreeNode(-1), TreeNode(-2))) == -1
print(max_path_sum(tree), max_path_sum(TreeNode(-3, TreeNode(-1), TreeNode(-2))))

25 -1


## 12.4 Search Patterns: Graphs, Heaps and Backtracking

In [ ]:
import heapq
from collections import deque, Counter

def bfs_dist(graph, start):              # BFS: distance in steps from start to every node
    dist = {start: 0}
    q = deque([start])
    while q:
        u = q.popleft()
        for v in graph[u]:
            if v not in dist:            # first visit is the shortest in an unweighted graph
                dist[v] = dist[u] + 1
                q.append(v)
    return dist

def top_k_frequent(items, k):            # heap: keep the k biggest counts
    return [w for w, _ in heapq.nlargest(k, sorted(Counter(items).items()), key=lambda p: p[1])]

def subsets(nums):                       # backtracking: choose, explore, un-choose
    out, path = [], []
    def go(start):
        out.append(path[:])              # every node of the recursion tree is an answer
        for i in range(start, len(nums)):
            path.append(nums[i])         # choose
            go(i + 1)                    # explore
            path.pop()                   # un-choose
    go(0)
    return out

g = {0: [1, 2], 1: [3], 2: [3, 4], 3: [5], 4: [5], 5: []}
assert bfs_dist(g, 0) == {0: 0, 1: 1, 2: 1, 3: 2, 4: 2, 5: 3}
assert top_k_frequent("aabbbc", 2) == ["b", "a"]
assert subsets([1, 2, 3]) == [[], [1], [1, 2], [1, 2, 3], [1, 3], [2], [2, 3], [3]]
print(bfs_dist(g, 0), top_k_frequent("aabbbc", 2), len(subsets([1, 2, 3])))

{0: 0, 1: 1, 2: 1, 3: 2, 4: 2, 5: 3} ['b', 'a'] 8


In [ ]:
def num_islands(grid):
    rows, cols = len(grid), len(grid[0]) if grid else 0
    seen = set()
    def flood(r, c):
        if r < 0 or c < 0 or r >= rows or c >= cols: return
        if grid[r][c] != "1" or (r, c) in seen: return
        seen.add((r, c))
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            flood(r + dr, c + dc)
    count = 0
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == "1" and (r, c) not in seen:
                count += 1
                flood(r, c)
    return count

assert num_islands(["11000", "11000", "00100", "00011"]) == 3
assert num_islands([]) == 0 and num_islands(["000"]) == 0 and num_islands(["111"]) == 1
print(num_islands(["11000", "11000", "00100", "00011"]))

3


In [ ]:
def combination_sum(cands, target):
    cands = sorted(cands)
    out, path = [], []
    def go(start, remaining):
        if remaining == 0:
            out.append(path[:])
            return
        for i in range(start, len(cands)):
            if cands[i] > remaining:
                break                    # sorted, so all later ones are too big as well
            path.append(cands[i])
            go(i, remaining - cands[i])  # i, not i + 1: reuse is allowed
            path.pop()
    go(0, target)
    return out

assert combination_sum([2, 3, 6, 7], 7) == [[2, 2, 3], [7]]
assert combination_sum([2], 1) == []
assert combination_sum([2, 3, 5], 8) == [[2, 2, 2, 2], [2, 3, 3], [3, 5]]
print(combination_sum([2, 3, 6, 7], 7), combination_sum([2], 1))

[[2, 2, 3], [7]] []


In [ ]:
from collections import defaultdict, deque

def ladder_length(begin, end, words):
    if end not in words:
        return 0
    buckets = defaultdict(list)          # "h*t" -> words that match it
    for w in set(words) | {begin}:
        for i in range(len(w)):
            buckets[w[:i] + "*" + w[i + 1:]].append(w)
    q, seen = deque([(begin, 1)]), {begin}
    while q:
        word, steps = q.popleft()
        if word == end:
            return steps
        for i in range(len(word)):
            for nb in buckets[word[:i] + "*" + word[i + 1:]]:
                if nb not in seen:
                    seen.add(nb)
                    q.append((nb, steps + 1))
    return 0

words = ["hot", "dot", "dog", "lot", "log", "cog"]
assert ladder_length("hit", "cog", words) == 5
assert ladder_length("hit", "cog", words[:-1]) == 0
print(ladder_length("hit", "cog", words), ladder_length("hit", "cog", words[:-1]))

5 0


## 12.5 Optimisation Patterns: Dynamic Programming and Greedy

In [ ]:
def climb(n):                            # 1D DP: ways to reach step n using 1 or 2 steps
    dp = [1, 1] + [0] * (n - 1)          # dp[i] = ways to stand on step i
    for i in range(2, n + 1):
        dp[i] = dp[i - 1] + dp[i - 2]    # arrive from one step below or two below
    return dp[n]

def coin_change(coins, amount):          # unbounded knapsack: fewest coins for each amount
    INF = amount + 1
    dp = [0] + [INF] * amount            # dp[a] = fewest coins to make a
    for a in range(1, amount + 1):
        for c in coins:
            if c <= a:
                dp[a] = min(dp[a], dp[a - c] + 1)
    return dp[amount] if dp[amount] != INF else -1

def edit_distance(a, b):                 # 2D DP on two strings
    d = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(len(a) + 1): d[i][0] = i          # delete all i letters
    for j in range(len(b) + 1): d[0][j] = j          # insert all j letters
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            d[i][j] = min(d[i - 1][j] + 1,                       # delete a[i-1]
                          d[i][j - 1] + 1,                       # insert b[j-1]
                          d[i - 1][j - 1] + (a[i - 1] != b[j - 1]))   # replace (free if equal)
    return d[len(a)][len(b)]

assert climb(5) == 8 and coin_change([1, 2, 5], 11) == 3 and coin_change([2], 3) == -1
assert edit_distance("horse", "ros") == 3
print(climb(5), coin_change([1, 2, 5], 11), coin_change([2], 3), edit_distance("horse", "ros"))

8 3 -1 3


In [ ]:
def can_jump(a):
    reach = 0
    for i, jump in enumerate(a):
        if i > reach:                    # an unreachable index: stuck
            return False
        reach = max(reach, i + jump)
    return True

assert can_jump([2, 3, 1, 1, 4]) and not can_jump([3, 2, 1, 0, 4])
assert can_jump([0]) and not can_jump([0, 1])
print(can_jump([2, 3, 1, 1, 4]), can_jump([3, 2, 1, 0, 4]))

True False


## 12.6 The Map of 150 Classic Problems

This is your practice list. The titles are the common names used across interview prep; the one-line approaches are written in our own words. Solve them in order inside each group: all the easy ones first, then the mediums, then the hards. Do not read the approach before you have tried for 15 to 20 minutes. When you are stuck, read only the approach line, close it, and code the idea yourself. Chapters 2 to 11 teach the tools; the group names match the patterns above.

## 12.7 The Interview Routine and Three Mock Interviews

In [ ]:
def two_sum(nums, target):
    index_of = {}                        # value -> position where we saw it
    for i, x in enumerate(nums):
        if target - x in index_of:       # check BEFORE storing so one item is never used twice
            return (index_of[target - x], i)
        index_of[x] = i
    return None

assert two_sum([2, 7, 11, 15], 9) == (0, 1)
assert two_sum([3, 3], 6) == (0, 1)
assert two_sum([-1, -2, -3, -4], -6) == (1, 3)
assert two_sum([1, 2], 10) is None
print(two_sum([2, 7, 11, 15], 9), two_sum([3, 3], 6), two_sum([-1, -2, -3, -4], -6))

(0, 1) (0, 1) (1, 3)


In [ ]:
def merge_intervals(ranges):
    merged = []
    for start, end in sorted(ranges):            # sort by start
        if merged and start <= merged[-1][1]:    # touches or overlaps the last block
            merged[-1][1] = max(merged[-1][1], end)
        else:
            merged.append([start, end])
    return merged

assert merge_intervals([[1, 3], [2, 6], [8, 10]]) == [[1, 6], [8, 10]]
assert merge_intervals([[1, 4], [4, 5]]) == [[1, 5]]
assert merge_intervals([[1, 10], [2, 3]]) == [[1, 10]]
assert merge_intervals([]) == []
assert merge_intervals([[5, 6], [1, 2]]) == [[1, 2], [5, 6]]
print(merge_intervals([[1, 3], [2, 6], [8, 10]]), merge_intervals([[1, 10], [2, 3]]))

[[1, 6], [8, 10]] [[1, 10]]


In [ ]:
from collections import deque

def task_order(n, pairs):
    after = [[] for _ in range(n)]       # after[b] = tasks that need b first
    waiting = [0] * n                    # unfinished prerequisites of each task
    for a, b in pairs:
        after[b].append(a)
        waiting[a] += 1
    ready = deque(i for i in range(n) if waiting[i] == 0)
    order = []
    while ready:
        t = ready.popleft()
        order.append(t)
        for nxt in after[t]:
            waiting[nxt] -= 1
            if waiting[nxt] == 0:
                ready.append(nxt)
    return order if len(order) == n else []      # fewer than n finished means a cycle

assert task_order(4, [[1, 0], [2, 0], [3, 1], [3, 2]]) == [0, 1, 2, 3]
assert task_order(2, [[0, 1], [1, 0]]) == []
assert task_order(1, [[0, 0]]) == []
assert task_order(3, []) == [0, 1, 2]
print(task_order(4, [[1, 0], [2, 0], [3, 1], [3, 2]]), task_order(2, [[0, 1], [1, 0]]))

[0, 1, 2, 3] []


## 12.8 Communication, Time and Failure Modes

## 12.9 A System for Spaced Repetition

In [ ]:
from datetime import date, timedelta

GAPS = [1, 3, 7, 14, 30]                  # days until the next review after each success

def next_review(solved_on, successes):
    """Date of the next review after `successes` clean recalls."""
    if successes >= len(GAPS):
        return None                       # finished: the problem is retired
    return solved_on + timedelta(days=GAPS[successes])

def due(log, today):
    return sorted(title for title, solved_on, successes in log
                  if (nr := next_review(solved_on, successes)) is not None and nr <= today)

log = [("Two Sum", date(2026, 1, 1), 0),
       ("Daily Temperatures", date(2026, 1, 1), 1),
       ("Edit Distance", date(2026, 1, 5), 0),
       ("Word Ladder", date(2026, 1, 8), 5)]
today = date(2026, 1, 5)
assert next_review(date(2026, 1, 1), 0) == date(2026, 1, 2)
assert due(log, today) == ["Daily Temperatures", "Two Sum"]
assert next_review(date(2026, 1, 8), 5) is None
print(due(log, today), next_review(date(2026, 1, 1), 2))

['Daily Temperatures', 'Two Sum'] 2026-01-08


## 12.10 The 12-Week Schedule

The plan assumes about 90 minutes a day, five days a week. Each week learns one topic (read the chapter, run the templates, solve the matching problems from the map in 12.6) and keeps the earlier topics alive through spaced review. The "new problems" column adds up to the 150 in the map.

## 12.11 How DSA Connects to Data Engineering and ML Interviews

Data roles still ask DSA, usually with a data flavour and often alongside SQL and Python questions. The patterns are the same; the story is different.

In [ ]:
import heapq

def sessionize(timestamps, gap):
    """Group sorted-or-unsorted times (in minutes) into sessions split by gaps larger than `gap`."""
    sessions = []
    for t in sorted(timestamps):
        if sessions and t - sessions[-1][-1] <= gap:
            sessions[-1].append(t)           # close enough: same session
        else:
            sessions.append([t])             # long silence: new session
    return sessions

def top_k_stream(stream, k):
    heap = []                                # min-heap of the k best (score, name) pairs
    for item in stream:
        if len(heap) < k:
            heapq.heappush(heap, item)
        elif item > heap[0]:
            heapq.heapreplace(heap, item)    # drop the weakest, add the newcomer
    return sorted(heap, reverse=True)

assert sessionize([1, 2, 3, 40, 41, 90], 30) == [[1, 2, 3], [40, 41], [90]]
stream = [(5, "a"), (9, "b"), (2, "c"), (7, "d"), (9, "e"), (1, "f")]
assert top_k_stream(stream, 3) == [(9, "e"), (9, "b"), (7, "d")]
print(sessionize([1, 2, 3, 40, 41, 90], 30), top_k_stream(stream, 3))

[[1, 2, 3], [40, 41], [90]] [(9, 'e'), (9, 'b'), (7, 'd')]


## 12.12 Self-Test: 20 Questions

Cover the answers. Give yourself 30 seconds per question. Write your answer down, then check. Score one point per correct answer; for questions that ask for a pattern, name the pattern and the signal for the point.

## Chapter recap

## Practice set

Twenty-two problems, graded from easy to hard. Try each one on paper first. Where a code block comes with a printed result, cover the output and predict it.

In [ ]:
from collections import Counter
print(Counter("banana").most_common(2))

[('a', 3), ('n', 2)]


In [ ]:
def is_palindrome(s):
    l, r = 0, len(s) - 1
    while l < r:
        if not s[l].isalnum():
            l += 1
        elif not s[r].isalnum():
            r -= 1
        elif s[l].lower() != s[r].lower():
            return False
        else:
            l += 1
            r -= 1
    return True

assert is_palindrome("A man, a plan, a canal: Panama")
assert not is_palindrome("race a car") and is_palindrome("")
print(is_palindrome("A man, a plan, a canal: Panama"), is_palindrome("race a car"))

True False


In [ ]:
def lower_bound_bad(a, target):
    lo, hi = 0, len(a)
    steps = 0
    while lo < hi and steps < 10:        # the steps guard only exists so this demo ends
        mid = (lo + hi) // 2
        if a[mid] < target:
            lo = mid                     # BUG: when mid == lo this never advances
        else:
            hi = mid
        steps += 1
    return lo, steps

print(lower_bound_bad([1, 3, 5, 7], 4))

(1, 10)


In [ ]:
a = [5, 1, 4, 2, 8]
pre = [0]
for x in a:
    pre.append(pre[-1] + x)
print(pre, pre[4] - pre[1])

[0, 5, 6, 10, 12, 20] 7


In [ ]:
grid = [[0] * 3] * 2
grid[0][0] = 1
print(grid)
good = [[0] * 3 for _ in range(2)]
good[0][0] = 1
print(good)

[[1, 0, 0], [1, 0, 0]]
[[1, 0, 0], [0, 0, 0]]


In [ ]:
from collections import Counter

def is_anagram(a, b):
    return Counter(a) == Counter(b)

assert is_anagram("listen", "silent") and not is_anagram("aab", "abb") and not is_anagram("ab", "abc")
print(is_anagram("listen", "silent"), is_anagram("aab", "abb"))

True False


In [ ]:
def best_window_bad(a, k):
    total = sum(a[:k])
    best = total
    for i in range(k, len(a) - 1):       # BUG: the last item is never added
        total += a[i] - a[i - k]
        best = max(best, total)
    return best

def best_window(a, k):
    total = sum(a[:k])
    best = total
    for i in range(k, len(a)):
        total += a[i] - a[i - k]
        best = max(best, total)
    return best

a = [1, 2, 3, 1, 4, 9]
assert best_window(a, 3) == 14
print(best_window_bad(a, 3), best_window(a, 3))

8 14


In [ ]:
def remove_adjacent_duplicates(s):
    stack = []
    for ch in s:
        if stack and stack[-1] == ch:
            stack.pop()                  # the pair cancels
        else:
            stack.append(ch)
    return "".join(stack)

assert remove_adjacent_duplicates("abbaca") == "ca"
assert remove_adjacent_duplicates("aabb") == ""
print(remove_adjacent_duplicates("abbaca"))

ca


In [ ]:
class ListNode:
    def __init__(self, val, nxt=None):
        self.val, self.next = val, nxt

def middle(head):
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
    return slow.val

four = ListNode(1, ListNode(2, ListNode(3, ListNode(4))))
five = ListNode(1, ListNode(2, ListNode(3, ListNode(4, ListNode(5)))))
assert middle(four) == 3 and middle(five) == 3
print(middle(four), middle(five))

3 3


In [ ]:
from collections import deque

def maze_steps(maze):
    R, C = len(maze), len(maze[0])
    if maze[0][0] == "#" or maze[R - 1][C - 1] == "#":
        return -1
    q, seen = deque([(0, 0, 0)]), {(0, 0)}
    while q:
        r, c, d = q.popleft()
        if (r, c) == (R - 1, C - 1):
            return d
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            nr, nc = r + dr, c + dc
            if 0 <= nr < R and 0 <= nc < C and maze[nr][nc] == "." and (nr, nc) not in seen:
                seen.add((nr, nc))       # mark on push
                q.append((nr, nc, d + 1))
    return -1

assert maze_steps(["..#", "#..", "#.."]) == 4
assert maze_steps([".#", "#."]) == -1
print(maze_steps(["..#", "#..", "#.."]), maze_steps([".#", "#."]))

4 -1


In [ ]:
def subsets_bad(nums):
    out, path = [], []
    def go(start):
        out.append(path)                 # BUG: stores the live list, not a copy
        for i in range(start, len(nums)):
            path.append(nums[i])
            go(i + 1)
            path.pop()
    go(0)
    return out

print(subsets_bad([1, 2]))

[[], [], [], []]


In [ ]:
from functools import lru_cache

calls = 0
def fib(n):
    global calls
    calls += 1
    return n if n < 2 else fib(n - 1) + fib(n - 2)

fib(5)
plain = calls

calls = 0
@lru_cache(maxsize=None)
def fib2(n):
    global calls
    calls += 1
    return n if n < 2 else fib2(n - 1) + fib2(n - 2)

fib2(5)
print(plain, calls)

15 6


In [ ]:
import heapq

class KthLargest:
    def __init__(self, k, nums):
        self.k, self.heap = k, []
        for x in nums:
            self.add(x)

    def add(self, x):
        if len(self.heap) < self.k:
            heapq.heappush(self.heap, x)
        elif x > self.heap[0]:
            heapq.heapreplace(self.heap, x)
        return self.heap[0]              # the smallest of the k biggest is the k-th largest

kl = KthLargest(3, [4, 5, 8, 2])
assert [kl.add(x) for x in (3, 5, 10, 9, 4)] == [4, 5, 5, 8, 8]
print([KthLargest(3, [4, 5, 8, 2]).add(3)], kl.heap[0])

[4] 8


In [ ]:
def rob(money):
    take, skip = 0, 0                    # best total if we take / skip the previous house
    for m in money:
        take, skip = skip + m, max(take, skip)
    return max(take, skip)

assert rob([2, 7, 9, 3, 1]) == 12 and rob([]) == 0 and rob([5]) == 5 and rob([1, 2]) == 2
print(rob([2, 7, 9, 3, 1]))

12


In [ ]:
from bisect import bisect_left

def lis_length(a):
    tails = []                           # tails[i] = smallest possible end of an increasing run of length i+1
    for x in a:
        i = bisect_left(tails, x)
        if i == len(tails):
            tails.append(x)
        else:
            tails[i] = x
    return len(tails)

assert lis_length([10, 9, 2, 5, 3, 7, 101, 18]) == 4
assert lis_length([]) == 0 and lis_length([5, 5, 5]) == 1
print(lis_length([10, 9, 2, 5, 3, 7, 101, 18]))

4


In [ ]:
import heapq

def min_rooms(meetings):
    ends = []                            # min-heap of end times of rooms in use
    for start, end in sorted(meetings):
        if ends and ends[0] <= start:
            heapq.heapreplace(ends, end) # reuse the room that frees up first
        else:
            heapq.heappush(ends, end)    # need a new room
    return len(ends)

assert min_rooms([[0, 30], [5, 10], [15, 20]]) == 2
assert min_rooms([[7, 10], [2, 4]]) == 1 and min_rooms([]) == 0
print(min_rooms([[0, 30], [5, 10], [15, 20]]))

2


In [ ]:
def greedy_coins(coins, amount):
    count = 0
    for c in sorted(coins, reverse=True):
        count += amount // c
        amount %= c
    return count if amount == 0 else -1

def dp_coins(coins, amount):
    dp = [0] + [float("inf")] * amount
    for a in range(1, amount + 1):
        dp[a] = min((dp[a - c] + 1 for c in coins if c <= a), default=float("inf"))
    return dp[amount]

assert greedy_coins([1, 3, 4], 6) == 3 and dp_coins([1, 3, 4], 6) == 2
print(greedy_coins([1, 3, 4], 6), dp_coins([1, 3, 4], 6))

3 2


In [ ]:
def trap(h):
    l, r = 0, len(h) - 1
    left_max = right_max = water = 0
    while l < r:
        if h[l] < h[r]:                  # the shorter side decides: its water level is its own max
            left_max = max(left_max, h[l])
            water += left_max - h[l]
            l += 1
        else:
            right_max = max(right_max, h[r])
            water += right_max - h[r]
            r -= 1
    return water

assert trap([0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1]) == 6
assert trap([]) == 0 and trap([3, 0, 3]) == 3
print(trap([0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1]))

6


In [ ]:
def largest_rectangle(h):
    stack = []                           # indexes with increasing heights
    best = 0
    for i, cur in enumerate(h + [0]):    # a final 0 forces every bar off the stack
        while stack and h[stack[-1]] >= cur:
            height = h[stack.pop()]
            left = stack[-1] + 1 if stack else 0
            best = max(best, height * (i - left))
        stack.append(i)
    return best

assert largest_rectangle([2, 1, 5, 6, 2, 3]) == 10
assert largest_rectangle([2, 4]) == 4 and largest_rectangle([]) == 0
print(largest_rectangle([2, 1, 5, 6, 2, 3]))

10


## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/dsa/ch12.html